# Grafo y regla de la cadena

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_multilayer-perceptrons/backprop.ipynb` · [Lección original](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Propagación hacia adelante, retropropagación y grafos computacionales
<a id="sec_backprop"></a>

Hasta ahora, hemos entrenado a nuestros modelos con descenso por gradiente estocástico minibatch. Sin embargo, cuando implementamos el algoritmo, sólo nos preocupaban los cálculos involucrados en la propagación hacia delante a través del modelo. Cuando llegó el momento de calcular los gradientes, simplemente invocamos la función de retropropagación proporcionada por el biblioteca de aprendizaje profundo.

El cálculo automático de gradientes simplifica profundamente la implementación de algoritmos de aprendizaje profundo. Antes de la diferenciación automática, incluso pequeños cambios en modelos complicados requerían recalcular derivadas complicados a mano. Sorprendentemente a menudo, los documentos académicos tenían que asignar numerosas páginas a la obtención de reglas de actualización. Si bien debemos seguir confiando en la diferenciación automática para que podamos centrarnos en las partes interesantes, usted debe saber cómo estos gradientes se calculan bajo el capó si desea ir más allá de una comprensión superficial del aprendizaje profundo.

En esta sección, tomamos una profunda inmersión en los detalles de * retropropagación * (más comúnmente llamado * retropropagación *). Para transmitir un poco de comprensión tanto para las técnicas y sus implementaciones, nos basamos en algunas matemáticas básicas y grafos computacionales. Para empezar, centramos nuestra exposición en un MLP de una capa oculta con deterioro de peso ($\ell_2$ regularización, que se describe en capítulos posteriores).

## Propagación hacia adelante
*Propagación hacia adelante* (o *paso hacia adelante*) se refiere al cálculo y almacenamiento de variables intermedias (incluyendo salidas) para una red neuronal en orden de la capa de entrada a la capa de salida. Ahora trabajamos paso a paso a través de la mecánica de una red neuronal con una capa oculta. Esto puede parecer tedioso, pero en las palabras eternas del funk virtuoso James Brown, usted debe "pagar el costo para ser el jefe".

Por el bien de la simplicidad, supongamos que el ejemplo de entrada es $\mathbf{x}\in \mathbb{R}^d$ y que nuestra capa oculta no incluye un término de sesgo. Aquí la variable intermedia es:

$$\mathbf{z}= \mathbf{W}^{(1)} \mathbf{x},$$

donde $\mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ es el parámetro de peso de la capa oculta. Después de ejecutar la variable intermedia $\mathbf{z}\in \mathbb{R}^h$ a través de la función de activación $\phi$ obtenemos nuestro vector de activación oculta de longitud $h$:

$$\mathbf{h}= \phi (\mathbf{z}).$$

La salida de capa oculta $\mathbf{h}$ es también una variable intermedia. Suponiendo que los parámetros de la capa de salida poseen sólo un peso de $\mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$, podemos obtener una variable de capa de salida con un vector de longitud $q$:

$$\mathbf{o}= \mathbf{W}^{(2)} \mathbf{h}.$$

Asumiendo que la función de pérdida es $l$ y la etiqueta de ejemplo es $y$, podemos calcular el término de pérdida para un solo ejemplo de datos,

$$L = l(\mathbf{o}, y).$$

Como veremos la definición de regularización $\ell_2$ que se introducirá más adelante, dado el hiperparametro $\lambda$, el término de regularización es

$$s = \frac{\lambda}{2} \left(\|\mathbf{W}^{(1)}\|_\textrm{F}^2 + \|\mathbf{W}^{(2)}\|_\textrm{F}^2\right),$$

:eqlabel:`eq_forward-s`

donde la norma Frobenius de la matriz es simplemente la norma $\ell_2$ aplicada después de aplanar la matriz en un vector. Finalmente, la pérdida regularizada del modelo en un ejemplo de datos dado es:

$$J = L + s.$$

Nos referimos a $J$ como la función *objetiva* en la siguiente discusión.

## Grafo computacional de la propagación hacia delante
El trazado de grafos computacionales nos ayuda a visualizar las dependencias de los operadores y variables dentro del cálculo.
[Referencia fig_forward](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html#fig-forward) contiene el grafo asociado
con la red simple descrita anteriormente, donde los cuadrados denotan variables y círculos denotan operadores. La esquina inferior izquierda significa la entrada y la esquina superior derecha es la salida. Observe que las direcciones de las flechas (que ilustran el flujo de datos) son principalmente derecha y hacia arriba.

![Grafo computacional de la propagación hacia delante.](../recursos/originales/forward.svg)
<a id="fig_forward"></a>

## Backpropagation
*Backpropagation* se refiere al método de cálculo del gradiente de los parámetros de la red neural. En resumen, el método atraviesa la red en orden inverso, desde la salida hasta la capa de entrada, de acuerdo con la regla *chain* del cálculo. El algoritmo almacena cualquier variable intermedia (derivados parciales) requerida mientras se calcula el gradiente con respecto a algunos parámetros. Supongamos que tenemos funciones $\mathsf{Y}=f(\mathsf{X})$ y $\mathsf{Z}=g(\mathsf{Y})$, en las que la entrada y la salida $\mathsf{X}, \mathsf{Y}, \mathsf{Z}$ son tensores de formas arbitrarias. Utilizando la regla de cadena, podemos calcular la derivada de $\mathsf{Z}$ con respecto a $\mathsf{X}$ vía

$$\frac{\partial \mathsf{Z}}{\partial \mathsf{X}} = \textrm{prod}\left(\frac{\partial \mathsf{Z}}{\partial \mathsf{Y}}, \frac{\partial \mathsf{Y}}{\partial \mathsf{X}}\right).$$

Aquí usamos el operador $\textrm{prod}$ para multiplicar sus argumentos después de que las operaciones necesarias, tales como la transposición y posiciones de entrada de intercambio, se han llevado a cabo. Para los vectores, esto es sencillo: es simplemente matriz--multiplicación de matriz. Para los tensores dimensionales superiores, utilizamos la contraparte apropiada. El operador $\textrm{prod}$ oculta todos los gastos generales de notación.

Recordemos que los parámetros de la red simple con una capa oculta, cuyo grafo computacional está en [Referencia fig_forward](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html#fig-forward), son $\mathbf{W}^{(1)}$ y $\mathbf{W}^{(2)}$. El objetivo de la retropropagación es calcular los gradientes $\partial J/\partial \mathbf{W}^{(1)}$ y $\partial J/\partial \mathbf{W}^{(2)}$. Para ello, aplicamos la regla de cadena y calculamos, a su vez, el gradiente de cada variable intermedia y parámetro. El orden de los cálculos se invierte en relación con los realizados en propagación hacia delante, ya que necesitamos comenzar con el resultado de la gráfica computacional y trabajar nuestro camino hacia los parámetros. El primer paso es calcular los gradientes de la función objetiva $J=L+s$ con respecto al término de pérdida $L$ y el término de regularización $s$:

$$\frac{\partial J}{\partial L} = 1 \; \textrm{and} \; \frac{\partial J}{\partial s} = 1.$$

A continuación, calculamos el gradiente de la función objetivo con respecto a la variable de la capa de salida $\mathbf{o}$ de acuerdo con la regla de cadena:

$$
\frac{\partial J}{\partial \mathbf{o}}
= \textrm{prod}\left(\frac{\partial J}{\partial L}, \frac{\partial L}{\partial \mathbf{o}}\right)
= \frac{\partial L}{\partial \mathbf{o}}
\in \mathbb{R}^q.
$$

A continuación, calculamos los gradientes del término de regularización con respecto a ambos parámetros:

$$\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)}
\; \textrm{and} \;
\frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}.$$

Ahora podemos calcular el gradiente $\partial J/\partial \mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$ de los parámetros del modelo más cercanos a la capa de salida.

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}}= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(2)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(2)}}\right)= \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}.$$

:eqlabel:`eq_backprop-J-h`

Para obtener el gradiente con respecto a $\mathbf{W}^{(1)}$ necesitamos continuar la retropropagación a lo largo de la capa de salida a la capa oculta. El gradiente con respecto a la salida de capa oculta $\partial J/\partial \mathbf{h} \in \mathbb{R}^h$ es dado por

$$
\frac{\partial J}{\partial \mathbf{h}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{h}}\right)
= {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}}.
$$

Dado que la función de activación $\phi$ se aplica a los elementos, el cálculo del gradiente $\partial J/\partial \mathbf{z} \in \mathbb{R}^h$ de la variable intermedia $\mathbf{z}$ requiere que utilicemos el operador de multiplicación de elementos, que denotamos por $\odot$:

$$
\frac{\partial J}{\partial \mathbf{z}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{h}}, \frac{\partial \mathbf{h}}{\partial \mathbf{z}}\right)
= \frac{\partial J}{\partial \mathbf{h}} \odot \phi'\left(\mathbf{z}\right).
$$

Finalmente, podemos obtener el gradiente $\partial J/\partial \mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ de los parámetros del modelo más cercanos a la capa de entrada.

$$
\frac{\partial J}{\partial \mathbf{W}^{(1)}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(1)}}\right)
= \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}.
$$

## Entrenamiento de redes neuronales
Cuando entrenamos redes neuronales, la propagación hacia delante y hacia atrás dependen unas de otras. En particular, para la propagación hacia delante, atravesamos el grafo computacional en la dirección de las dependencias y calculamos todas las variables en su trayectoria. Estos se utilizan entonces para la retropropagación donde el orden de cálculo en el gráfico se invierte.

Por un lado, computar el término de regularización [Referencia eq_forward-s](https://d2l.ai/#eq-forward-s) durante la propagación hacia delante depende de los valores actuales de los parámetros modelo $\mathbf{W}^{(1)}$ y $\mathbf{W}^{(2)}$. Son dados por el algoritmo de optimización según la retropropagación en la iteración más reciente. Por otro lado, el cálculo de gradiente para el parámetro
[Referencia eq_backprop-J-h](https://d2l.ai/#eq-backprop-J-h) durante la retropropagación
depende del valor actual de la capa oculta salida $\mathbf{h}$, que se da por propagación hacia delante.

Por lo tanto, cuando entrenamos redes neuronales, una vez iniciados los parámetros del modelo, alternamos la propagación hacia delante con la retropropagación, actualizando los parámetros del modelo utilizando gradientes dados por retropropagación. Tenga en cuenta que la retropropagación reutiliza los valores intermedios almacenados de propagación hacia delante para evitar cálculos duplicados. Una de las consecuencias es que necesitamos retener los valores intermedios hasta que se complete la retropropagación. Esta es también una de las razones por las que el entrenamiento requiere significativamente más memoria que una predicción simple. Además, el tamaño de tales valores intermedios es aproximadamente proporcional al número de capas de red y al tamaño del lote. Así, entrenar redes más profundas utilizando tamaños de lote más grandes conduce a errores de *fuera de la memoria*.

## Resumen
La propagación hacia delante calcula y almacena secuencialmente variables intermedias dentro del grafo computacional definido por la red neuronal. Proviene de la entrada a la capa de salida. La retropropagación calcula y almacena secuencialmente los gradientes de variables intermedias y parámetros dentro de la red neuronal en el orden inverso. Al entrenar modelos de aprendizaje profundo, la propagación hacia delante y la retropropagación son interdependientes, y el entrenamiento requiere significativamente más memoria que predicción.



### Nota docente de Hespérides

Sigue tres objetos diferentes: el valor de la pérdida, su gradiente y la actualización que calcula el optimizador. Comprueba formas y reinicia los gradientes antes de cada paso. En el explorador se mantienen función y punto inicial para comparar trayectorias; una misma tasa no significa el mismo desplazamiento efectivo para todos los métodos.

Vínculo con los apuntes: sesión 2, «Grafo y regla de la cadena».


## Ejercicios
1. Supongamos que las entradas $\mathbf{X}$ a alguna función escalar $f$ son matrices $n \times m$. ¿Cuál es la dimensión del gradiente de $f$ con respecto a $\mathbf{X}$?
1. Añadir un sesgo a la capa oculta del modelo descrito en esta sección (no es necesario incluir el sesgo en el término regularización).
    1. Dibuje el grafo computacional correspondiente.
    1. Derivar las ecuaciones de propagación hacia delante y hacia atrás.
1. Calcular la huella de memoria para entrenamiento y predicción en el modelo descrito en esta sección.
1. Supón que deseas calcular las segundas derivadas. ¿Qué sucede con el grafo computacional? ¿Cuánto tiempo esperas que requiera el cálculo?
1. Supongamos que el grafo computacional es demasiado grande para su GPU.
    1. ¿Se puede dividir en más de una GPU?
    1. ¿Cuáles son las ventajas y desventajas sobre el entrenamiento en un minibatch más pequeño?

[Debate del original](https://discuss.d2l.ai/t/102)
